In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers,models
from tensorflow.keras.applications import MobileNetV2
data,info=tfds.load('plant_village',split='train',with_info=True,as_supervised=True)

I0000 00:00:1782669084.783028    1065 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782669087.825352    1065 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782669091.651936    1065 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782669098.166444    1065 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 

In [2]:
data_size=info.splits["train"].num_examples
class_names=info.features["label"].names
n_classes=info.features["label"].num_classes
print("data size :",data_size,"\nclass_names:",class_names,"\n classes :",n_classes)

data size : 54303 
class_names: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry___healthy', 'Cherry___Powdery_mildew', 'Corn___Cercospora_leaf_spot Gray_leaf_spot', 'Corn___Common_rust', 'Corn___healthy', 'Corn___Northern_Leaf_Blight', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___healthy', 'Strawberry___Leaf_scorch', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___healthy', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___

In [3]:
for image, label in data.take(1):
    print('Image shape:', image.shape)
    print('Label:', label.numpy())

I0000 00:00:1782669098.490624    1212 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


Image shape: (256, 256, 3)
Label: 35


In [4]:
for image, label in data.take(1):
    print('Image shape:', image.shape)
    print('Label:', label.numpy())

Image shape: (256, 256, 3)
Label: 35


In [5]:
train_data=data.take(round(0.8*data_size))
test_data=data.skip(round(0.2*data_size))

In [6]:
AUTOTUNE = tf.data.AUTOTUNE
def preprocess(image,label):
    image=tf.image.resize(image,(224,224))
    image=tf.cast(image,tf.float32)/224.0
    label=tf.one_hot(label,depth=38)
    return image,label
train_p_data=train_data.map(preprocess, tf.data.AUTOTUNE)
test_p_data=test_data.map(preprocess, tf.data.AUTOTUNE)

In [7]:
train_p_data = (train_p_data.shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE))
test_p_data = (test_p_data.batch(32).prefetch(tf.data.AUTOTUNE))

In [8]:
for image, label in train_p_data.take(1):
    print('Image shape:', image.shape)
    print('Label:', label.numpy())

W0000 00:00:1782669100.450906    1383 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 19272448 bytes after encountering the first element of size 19272448 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
W0000 00:00:1782669100.456191    1065 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 19272448 bytes after encountering the first element of size 19272448 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


Image shape: (32, 224, 224, 3)
Label: [[0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [9]:
augumentation=tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(.2),
    layers.RandomZoom(.2),
    layers.RandomBrightness(.2)
])
train_p_data=train_p_data.map(lambda image,label:(augumentation(image,training=True),label),
                              num_parallel_calls=AUTOTUNE)

MobileNetV2 model

In [10]:
def build_model(model_backbone,train_base=False):
    base=model_backbone(input_shape=(224,224,3),include_top=False,weights="imagenet")

    base.trainble=train_base
    inputs=tf.keras.Input(shape=(224,224,3))
    x=base(inputs,training=False)
    x=layers.GlobalAveragePooling2D()(x)
    x=layers.Dense(256,activation='relu')(x)
    x=layers.Dropout(.4)(x)
    outputs=layers.Dense(38,activation='softmax')(x)
    return models.Model(inputs,outputs),base

model,base=build_model(MobileNetV2)
model.compile(optimizer=tf.keras.optimizers.Adam(.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary() 

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         9,766 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,595,686 (9.90 MB)

 Trainable params: 2,561,574 (9.77 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [ ]:
history1 = model.fit(
    train_p_data,
    validation_data=test_p_data,
    epochs=10)

Epoch 1/10


W0000 00:00:1782669123.722072    1535 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 19272448 bytes after encountering the first element of size 19272448 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
I0000 00:00:1782669127.769556    1195 service.cc:153] XLA service 0x75ebcc095ec0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782669127.775411    1195 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.2)
I0000 00:00:1782669148.304544    1195 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
